# KV Cache Compression Demo

This notebook demonstrates KV cache compression using trained RL agents. Compression is **emulated via attention masking**: evicted KV entries are masked out rather than physically removed from the cache. All original computations are still performed, plus some overhead from the masking logic.

**This demo does not provide any speedup.** It is intended purely to observe the effect of learned compression on generation quality. For actual inference speedups, the compression must be integrated at a lower level (e.g., by physically evicting entries from the KV cache).

**Prerequisites:**
1. Model weights in `models/Qwen2.5-7B-Instruct/`
2. Trained agents with config composite in `config_composites/my_sweep.yaml`
   - Generate with: `uv run python scripts/orchestration/assignment_generator.py write-composites my_sweep`

In [ ]:
import torch

from omegaconf import DictConfig, OmegaConf
from torchtune import config, training
from transformers import AutoTokenizer

from kvcompression import PROJECT_ROOT
from kvcompression.hooks.compressor import KVCompressor, LayerHeadCompressionConfig
from kvcompression.kv_cache.compression_strategy_protocol import (
    PressCompressionStrategy,
)
from kvcompression.utils.generation import generate_with_compression

# Configuration
MODEL_NAME = "Qwen2.5-7B-Instruct"
MODEL_DIR = PROJECT_ROOT / "models" / MODEL_NAME
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.bfloat16

print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model (standard torchtune model - compression works via monkey-patching)
model_cfg = DictConfig({"_component_": "torchtune.models.qwen2_5.qwen2_5_7b_instruct"})
checkpointer_cfg = DictConfig(
    {
        "_component_": "torchtune.training.FullModelHFCheckpointer",
        "checkpoint_dir": str(MODEL_DIR),
        "checkpoint_files": [
            "model-00001-of-00004.safetensors",
            "model-00002-of-00004.safetensors",
            "model-00003-of-00004.safetensors",
            "model-00004-of-00004.safetensors",
        ],
        "output_dir": "/tmp",
        "model_type": "QWEN2",
    }
)

print("Loading model weights...")
checkpointer = config.instantiate(checkpointer_cfg)
ckpt_dict = checkpointer.load_checkpoint()

with training.set_default_dtype(DTYPE), DEVICE:
    model = config.instantiate(model_cfg)

model.load_state_dict(ckpt_dict[training.MODEL_KEY], strict=True)
model = model.to(DEVICE)
model.eval()

print(f"Model loaded: {model}")

In [ ]:
def load_compression_configs(file_name: str, shared_options: dict = None):
    """Load compression configs from a config_composites file."""
    config_path = PROJECT_ROOT / "config_composites" / f"{file_name}.yaml"

    if not config_path.exists():
        print(f"Config file not found: {config_path}")
        print(
            "Generate one with: uv run python scripts/orchestration/assignment_generator.py write-composites your_sweep"
        )
        return []

    data = OmegaConf.load(config_path)

    compression_configs = []
    for assignment in data.assignments:
        layer = assignment.get("layer")
        head = assignment.get("head")

        press_config = assignment.press
        if shared_options:
            press_config = OmegaConf.merge(press_config, shared_options)

        with training.set_default_dtype(DTYPE), DEVICE:
            press = config.instantiate(press_config)
        press = press.to(DEVICE)

        compression_strategy = PressCompressionStrategy(press=press)
        compression_configs.append(
            LayerHeadCompressionConfig(
                layer=layer,
                head=head,
                strategy=compression_strategy,
                strategy_name=f"agent_L{layer}_H{head}",
            )
        )

    print(f"Loaded {len(compression_configs)} compression configs")
    return compression_configs


# Load trained agents from config composite
# Change "my_sweep" to your sweep name
SWEEP_NAME = "my_sweep"
compression_configs = load_compression_configs(
    SWEEP_NAME, shared_options={"n_sinks": 4, "n_running_window": 16}
)

In [ ]:
def generate_text(
    model,
    tokenizer,
    prompt: str,
    max_new_tokens: int = 50,
    compression_configs: list = None,
    kv_cache_size: int = 500,
) -> str:
    """Generate text with optional KV compression."""
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)
    total_length = input_ids.shape[1] + max_new_tokens

    print(f"  Tokens in cache after prefill: {input_ids.shape[1]}")

    if compression_configs:
        with KVCompressor(
            model=model, compression_strategies=compression_configs
        ) as kv_compressor:
            with DEVICE:
                model.setup_caches(
                    batch_size=1, dtype=DTYPE, decoder_max_seq_len=total_length
                )
            generated_tokens = generate_with_compression(
                model,
                input_ids,
                kv_compressor,
                kv_cache_size,
                max_new_tokens,
                stop_tokens=[tokenizer.eos_token_id]
                if tokenizer.eos_token_id
                else None,
            )
    else:
        # Without compression
        with DEVICE:
            model.setup_caches(
                batch_size=1, dtype=DTYPE, decoder_max_seq_len=total_length
            )
        generated_tokens = generate_with_compression(
            model,
            input_ids,
            None,
            0,
            max_new_tokens,
            stop_tokens=[tokenizer.eos_token_id] if tokenizer.eos_token_id else None,
        )
        model.reset_caches()

    return tokenizer.decode(generated_tokens[0], skip_special_tokens=True)

In [ ]:
# Test prompt
prompt = """The history of artificial intelligence began in antiquity, with myths and stories of artificial beings. The seeds of modern AI were planted by classical philosophers who attempted to describe human thinking as mechanical manipulation of symbols.

Question: What did classical philosophers try to describe?

Answer:"""

KV_CACHE_SIZE = 35  # Target cache size after compression

with torch.no_grad():
    # Generate without compression
    print("Without compression:")
    output_baseline = generate_text(model, tokenizer, prompt, max_new_tokens=50)
    print(output_baseline)

    print("\n" + "=" * 50 + "\n")

    # Generate with compression
    if compression_configs:
        print(f"With compression (cache_size={KV_CACHE_SIZE}):")
        output_compressed = generate_text(
            model,
            tokenizer,
            prompt,
            max_new_tokens=50,
            compression_configs=compression_configs,
            kv_cache_size=KV_CACHE_SIZE,
        )
        print(output_compressed)
    else:
        print("No compression configs loaded. Update SWEEP_NAME in cell 3.")